# MicroPlant — Model Training

This notebook trains the MicroPlant model for plant disease classification.

Goals:
- Train a lightweight CNN model
- Evaluate performance using F1-score
- Monitor training and validation metrics
- Prepare model for compression and deployment

In [9]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import matplotlib.pyplot as plt

from src.preprocessing import get_dataloaders
from src.architectures import get_microplant, get_teacher_model
from src.training import train_model, validate
from src.utils import count_parameters, count_model_bytes

## Configuration

We define:
- Dataset path
- Device (CPU / GPU)
- Training parameters

In [7]:
DATA_DIR = "../data/MicroPlantDataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 64
EPOCHS = 15
LR = 0.001

print(f"using: {DEVICE}")

using: cpu


## Load Data

We load the dataset using the predefined pipeline:
- Train / Validation / Test split
- Data augmentation applied to training set

In [4]:
train_loader, val_loader, test_loader, class_names = get_dataloaders(DATA_DIR, batch_size=BATCH_SIZE)

print("Classes:", class_names)

Classes: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy']


## Initialize Model

We use the MicroPlant architecture:
- Depthwise separable convolutions
- Squeeze-and-Excitation (SE)
- Lightweight design for edge deployment

In [5]:
model = get_microplant(num_classes=len(class_names)).to(DEVICE)

print("Total parameters:", count_parameters(model))
count_model_bytes(model)

Total parameters: 8804
Model Weights: 35,216 bytes
Model Buffers: 2,648 bytes
Total Size: 36.98 KB


37864

## Training

We train the model using:
- CrossEntropyLoss
- Adam optimizer
- F1-score as evaluation metric

In [8]:
model = train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=LR,
    save_name="../models/microplant",
    device=DEVICE
)

Training:   0%|          | 0/41 [00:00<?, ?it/s]c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Training:   0%|          | 0/41 [00:00<?, ?it/s]


AssertionError: Torch not compiled with CUDA enabled

## Evaluation

We evaluate the trained model on the test set.

In [10]:
test_loss, test_f1 = validate(model, test_loader, device=DEVICE)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

c:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test Loss: 1.3627
Test F1 Score: 0.3545


## Observations

- Training performance improves over epochs
- Validation F1-score indicates generalization ability
- Model is ready for compression and optimization